# Finance headline sentiment: vLLM serving and evaluation

Serves the base model and the DPO model with vLLM on both T4 GPUs, then scores 500 held-out headlines with a
deterministic checker: valid-JSON rate, accuracy and throughput. A word-list challenger flags the answers a person
should review, the same design as Reconcile.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!nvidia-smi
# uv picks the PyTorch build that matches this machine's GPU driver, as vLLM's install guide recommends.
!pip install -q uv
!timeout 900 uv pip install --system -q vllm datasets wandb --torch-backend=auto
# Kaggle's torchaudio is built for a different CUDA version, which stops vLLM from importing. vLLM doesn't use it.
!pip uninstall -y -q torchaudio
!python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.device_count(), 'GPUs')"
!pip list 2>/dev/null | grep -iE "^(torch|transformers|trl|peft|accelerate|deepspeed|vllm|wandb) "

In [ ]:
%%writefile finsent.py
"""Shared code for the financial-headline sentiment project.

The model must answer with JSON such as {"label": "bullish"}. A deterministic checker scores each answer,
and a word-list challenger flags the headlines a person should review, the same design as Reconcile.
"""
import json
import random
import re

LABELS = ["bearish", "bullish", "neutral"]  # dataset labels 0, 1 and 2
# Run 2 prompt. Run 1 showed the labels as {"label": "bearish" | "bullish" | "neutral"}, and the base model
# copied that pattern with a quote missing, so this version names the labels and gives one plain example.
PROMPT = ('Classify the sentiment of this financial news headline as bearish, bullish or neutral. '
          'Reply with JSON only, for example {"label": "neutral"}.\n\n')
PROMPT_RUN1 = ('Classify the sentiment of this financial news. '
               'Reply only with JSON: {"label": "bearish" | "bullish" | "neutral"}.\n\n')


def to_pair(example, index, rng=random):
    """Builds one DPO preference pair in conversational format.

    The rejected answer alternates between a wrong label in valid JSON (odd rows) and the right label
    wrapped in prose, which isn't valid JSON (even rows).
    """
    gold = LABELS[example["label"]]
    wrong = rng.choice([lab for lab in LABELS if lab != gold])
    rejected = json.dumps({"label": wrong}) if index % 2 else f"Sure! The sentiment is {gold}."
    return {
        "prompt": [{"role": "user", "content": PROMPT + example["text"]}],
        "chosen": [{"role": "assistant", "content": json.dumps({"label": gold})}],
        "rejected": [{"role": "assistant", "content": rejected}],
    }


def check(raw, gold):
    """Strict checker: the whole answer must parse as JSON with an allowed label."""
    try:
        obj = json.loads(raw.strip())
    except (json.JSONDecodeError, AttributeError):
        return {"valid_json": False, "allowed_label": False, "label": None, "correct": False}
    label = str(obj.get("label", "")).strip().lower() if isinstance(obj, dict) else ""
    allowed = label in LABELS
    return {"valid_json": True, "allowed_label": allowed, "label": label if allowed else None,
            "correct": allowed and label == gold}


BULLISH = ["beat", "beats", "raise", "raises", "raised", "upgrade", "upgrades", "upgraded", "surge", "surges",
           "soar", "soars", "jump", "jumps", "gain", "gains", "rally", "rallies", "record", "higher", "rise",
           "rises", "buy", "outperform", "bullish", "boost", "boosts", "strong", "tops", "climbs"]
BEARISH = ["miss", "misses", "missed", "cut", "cuts", "downgrade", "downgrades", "downgraded", "fall", "falls",
           "drop", "drops", "plunge", "plunges", "slump", "slumps", "tumble", "tumbles", "lower", "loss",
           "losses", "sell", "underperform", "bearish", "decline", "declines", "slide", "slides", "warns",
           "weak", "layoffs", "lawsuit"]
_WORD = re.compile(r"[a-z]+")


def rules_label(text):
    """Word-list challenger: more bullish words than bearish ones means bullish, and so on."""
    words = _WORD.findall(text.lower())
    score = sum(w in BULLISH for w in words) - sum(w in BEARISH for w in words)
    return "bullish" if score > 0 else "bearish" if score < 0 else "neutral"


def summarize(outputs, golds, texts):
    """Scores a model's raw answers. Disagreements with the challenger form the review queue."""
    rows = [check(o, g) for o, g in zip(outputs, golds)]
    n = len(rows)
    review = [r["label"] != rules_label(t) for r, t in zip(rows, texts)]
    wrong = [not r["correct"] for r in rows]
    caught = sum(w and q for w, q in zip(wrong, review))
    return {
        "n": n,
        "valid_json_rate": sum(r["valid_json"] for r in rows) / n,
        "accuracy": sum(r["correct"] for r in rows) / n,
        "review_rate": sum(review) / n,
        "errors": sum(wrong),
        "errors_in_review": caught,
    }

In [ ]:
%%writefile run_vllm.py
"""Serves one model with vLLM on the 500 held-out headlines and saves its raw answers and throughput.

Usage: python run_vllm.py <name> <model path or Hub id> <tensor_parallel_size>
Each model runs in its own process, so its GPU memory is fully released before the next one starts.
"""
import json
import os
import sys
import time

T0 = time.perf_counter()


def note(message):
    print(f"[{time.perf_counter() - T0:7.1f}s] {message}", flush=True)


note("importing vLLM")
from datasets import load_dataset  # noqa: E402
from vllm import LLM, SamplingParams  # noqa: E402

from finsent import LABELS, PROMPT  # noqa: E402

name, path, tp = sys.argv[1], sys.argv[2], int(sys.argv[3])
val = load_dataset("zeroshot/twitter-financial-news-sentiment")["validation"].shuffle(seed=42).select(range(500))
convs = [[{"role": "user", "content": PROMPT + t}] for t in val["text"]]

note(f"starting vLLM for {name} on {tp} GPU(s)")
# enforce_eager skips torch.compile and CUDA graph capture. Two earlier runs stalled at start-up on the T4s.
llm = LLM(model=path, dtype="half", tensor_parallel_size=tp, seed=0, enforce_eager=True,
          disable_custom_all_reduce=True)
note("model loaded")
params = SamplingParams(temperature=0, max_tokens=32)
llm.chat(convs[:16], params)  # warm-up, so the timing below leaves out start-up costs
note("warm-up done")
start = time.perf_counter()
outs = llm.chat(convs, params)
seconds = time.perf_counter() - start

with open(f"/kaggle/working/outputs_{name}.json", "w") as fh:
    json.dump({"model": name, "path": path, "tensor_parallel_size": tp, "enforce_eager": True,
               "seconds": seconds, "headlines_per_second": len(convs) / seconds,
               "outputs": [o.outputs[0].text for o in outs],
               "texts": list(val["text"]), "gold": [LABELS[g] for g in val["label"]]}, fh)
note(f"{name}: {len(convs)} headlines in {seconds:.2f}s on {tp} GPU(s)")
# vLLM's worker processes keep Python alive after the run, so the script exits directly once results are saved.
sys.stdout.flush()
os._exit(0)

In [ ]:
import glob, json, os, signal, subprocess

def run_limited(cmd, log_path, limit_seconds):
    """Runs cmd in its own process group with a hard time limit, writing its output to log_path."""
    with open(log_path, "w") as log:
        p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            code = p.wait(timeout=limit_seconds)
        except subprocess.TimeoutExpired:
            code = "timed out"
        try:
            os.killpg(p.pid, signal.SIGKILL)  # ends any vLLM worker processes still holding the GPUs
        except ProcessLookupError:
            pass
        p.wait()
    print(open(log_path, errors="replace").read()[-3000:])
    return code

config = glob.glob("/kaggle/input/**/qwen-dpo-merged/config.json", recursive=True)
assert config, "Add the training notebook's output as an input to this notebook."
models = {"base": "Qwen/Qwen2.5-0.5B-Instruct", "dpo": os.path.dirname(config[0])}

# Two T4s have no NVLink. Peer-to-peer over PCIe can stall NCCL on cloud VMs, so it is switched off.
os.environ["NCCL_P2P_DISABLE"] = "1"
for name, path in models.items():
    for tp in (2, 1):  # both GPUs first; one GPU if two-GPU serving fails or passes 10 minutes
        print(f"=== {name} with tensor_parallel_size={tp}")
        code = run_limited(["python", "run_vllm.py", name, path, str(tp)], f"/kaggle/working/vllm_{name}_tp{tp}.log", 600)
        print(f"--- exit: {code}")
        if code == 0:
            break
    else:
        raise RuntimeError(f"vLLM failed for {name}")

In [ ]:
import json
from finsent import summarize

results = {}
for name in ("base", "dpo"):
    d = json.load(open(f"/kaggle/working/outputs_{name}.json"))
    s = summarize(d["outputs"], d["gold"], d["texts"])
    s.update(headlines_per_second=d["headlines_per_second"], tensor_parallel_size=d["tensor_parallel_size"])
    results[name] = s
json.dump(results, open("/kaggle/working/results.json", "w"), indent=1)

print(f"{'':6}{'valid JSON':>12}{'accuracy':>10}{'review':>8}{'errors':>8}{'in review':>11}{'per sec':>9}")
for name, s in results.items():
    print(f"{name:6}{s['valid_json_rate']:>12.1%}{s['accuracy']:>10.1%}{s['review_rate']:>8.1%}"
          f"{s['errors']:>8}{s['errors_in_review']:>11}{s['headlines_per_second']:>9.1f}")

In [ ]:
import os, wandb

os.environ["WANDB_MODE"] = "offline"  # synced afterwards, so no API key is stored in the notebook
run = wandb.init(project="finsent-dpo", name="eval-500-headlines", dir="/kaggle/working")
cols = ["model", "valid_json_rate", "accuracy", "review_rate", "errors", "errors_in_review", "headlines_per_second"]
run.log({"results": wandb.Table(columns=cols, data=[[m] + [results[m][c] for c in cols[1:]] for m in results])})
run.finish()